# Calculate the spatial gradient of a field on the mesh

**Step 1: Create FELiCSMesh, FELiCSSpace and Field**



Sophie: Maybe it would be a good idea to do that in 2 chapters; one for cartesian, one for cylindrical coordinates. 

### Cartesian coordinates:

=> not so much explanation needed

=> Idea: maybe the spectral dimension could aready be introduced here? Then only the Neumann Boundary condition has to be explained later. The square mesh would become a channel.

### Cylindrical coordinates:

=> explain cylindrical coordinates (x,r,theta) and that the square becomes a cylinder

=> rename scalarField into "a" or something s.t. the equation looks nice

=> put equation of the field "a" with spectral factor (a * exp(1j\*m\*theta)) and explain wavenumber m

=> put equation of gradient with derivative w.r.t. spectral dimension (if any doubt, see https://en.wikipedia.org/wiki/Del_in_cylindrical_and_spherical_coordinates)

=> put validation field creation in its own cell and explain shortly (do we need it here?)

=> put a "TODO" note for later (for after the retreat): for the cylindrical coordinates, if there is a boundary with r=0, this axis is a symmetry boundary, thus there HAS to be either a zero Dirichlet or a zero Neumann condition for "a". We don't set any boundary conditions yet (because the Neumann handling will be implemented after the retreat), but here a Neumann condition should be set for both r and theta

In [1]:
from FELiCS.SpaceDisc.FELiCSMesh import FELiCSMesh

meshFileName = "./square_mesh.msh"
coordinateSystemName = "Cylindrical"
m = 3

# Create a FELiCS mesh from the gmsh file
mesh = FELiCSMesh(coordinateSystemName=coordinateSystemName, meshFileName=meshFileName)

from FELiCS.SpaceDisc.FEMSpaces import getFELiCSSpace

scalarSpace = getFELiCSSpace(mesh, order=2, dim=1)
vectorSpace = getFELiCSSpace(mesh, order=2, dim=3)

from FELiCS.Fields.Field import Field

scalarField = Field(scalarSpace, mesh=mesh, m = m)
scalarField.importH5("function_values_sine")
gradientFieldsol = Field(vectorSpace, mesh=mesh)
gradientFieldsol.importH5("grad_solution_cylSpectral_m=3")

fatal: not a git repository: '/home/frederic/miniforge3/envs/felics2025-env/lib/python3.13/site-packages/../.git'
Info     | FELiCSMesh.py          | __init__                   (line 28  ) : Opening mesh file: ./square_mesh.msh
Info     | FELiCSMesh.py          | __init__                   (line 36  ) : Mesh contains 513 nodes and 1024 elements


(         (               (     
)\ )      ) )       (    )\ )  
(()/(  (  (()/( (    )\   (()/(  
/(_)) )\  /(_)))\  (((_)  /(_)) 
(_)_)((_) (_)) ((_) )\___ (_))   
| __|| __|| |   (_)((/ __|/ __|  
| _| | _| | |__ | | | (__ \__ \  
|_|  |___||____||_|  \___||___/  
Git commit: 


**Step 2: Define the gradient Expression and evaluate it on the Field**
We now want to calculate the gradient of our scalar quantity using the `getGradient()` method of our Field class. This method returns a Field 

In [ ]:
##################################
# Tensor Utils stuff in Tensor Utils tutorial
from ufl import TestFunction, dx, conj
from FELiCS.Misc.tensorUtils import iGrad, iConj, iDot, Tensor
import dolfinx.fem

J_hat = mesh.coordinateSystem.J_hat
v = TestFunction(vectorSpace)
v_tens = Tensor(v, CoordSys=mesh.coordinateSystem, hasSpectralDimension=True)
expression = iDot(iGrad(scalarField.getTensor()), iConj(v_tens)).ufl_tens * J_hat* dx
###################################

gradientField = scalarField.getGradientField()

print(sum(gradientField.getCoefficientArray() - gradientFieldsol.getCoefficientArray()))

(-0.19815778792027344+92.17994219903049j)
